# Notebook Lengkap Eksperimen INDOMEME
## Kelompok 7 dan Joseph Lase
Pengumpulan 30 Oktober 2026. Bukti utama berasal dari eksperimen 18–29 Oktober 2026.

**Jika meme menggabungkan gambar dan teks, apakah menambahkan gambar selalu membantu model?**
Meme dapat menyampaikan kebencian atau ketidaklayakan melalui hubungan visual, teks, dan konteksnya.
Notebook ini mengikuti pertanyaan tersebut dari audit data sampai hasil terukur, termasuk hasil yang
tidak membaik dan percobaan yang tidak selesai.

Paket inti mencakup **8 konfigurasi supervised dan 10 hasil tugas**. Riwayat proyek juga menyimpan
14 baseline konvensional pada validation, hasil SwinV2 historis di luar paket inti, kalibrasi,
serta kegagalan yang dipisahkan dari eksperimen selesai. RQ3/API dan novelty tidak dinyatakan selesai.

**Mode eksekusi notebook:** membaca artefak yang dibekukan, menghitung ulang metrik, dan menggambar
visualisasi. Tidak ada training ulang, inferensi model, pemanggilan API, atau unduhan dataset/bobot.
Kode implementasi tetap diperlihatkan agar alur data, model, loss, dan pelatihan dapat dijelaskan.
Sel terakhir mencatat host dan waktu eksekusi aktual. Tidak ada keluaran yang diisi secara manual.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone, timedelta
import ast, hashlib, json, platform, socket, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Code
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

ROOT=next((p.resolve() for p in [Path.cwd(),Path.cwd().parent]
           if (p/'results/core_results.csv').is_file() and (p/'evidence/research_facts.json').is_file()),None)
assert ROOT is not None, 'Jalankan notebook dari root paket atau satu subfolder di dalamnya.'
pd.set_option('display.max_columns',20)
pd.set_option('display.max_colwidth',100)
plt.rcParams.update({'figure.dpi':115,'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
CHECKS={'training_or_inference_started':False,'core_test_outcomes':0,'baseline_validation_trials':0}

def path(relative):
    candidate=(ROOT/relative).resolve()
    assert candidate.is_relative_to(ROOT), 'Path keluar dari paket.'
    return candidate

def read_json(relative):
    return json.loads(path(relative).read_text(encoding='utf-8-sig'))

def read_csv(relative):
    return pd.read_csv(path(relative),keep_default_na=False)

def sha256(file):
    with Path(file).open('rb') as stream:
        return hashlib.file_digest(stream,'sha256').hexdigest()

facts=read_json('evidence/research_facts.json')
inventory=read_json('evidence/inventory.json')
audit=read_json('evidence/local_results_audit.json')
core=read_csv('results/core_results.csv')
assert audit['passed'] and len(core)==10 and core.id.nunique()==8
print('Root paket:',ROOT)
print('Python:',platform.python_version(),'pandas:',pd.__version__,'numpy:',np.__version__)
print('Mode: analisis artefak tersimpan, tanpa training atau inferensi baru.')

## Pertanyaan riset dan cara membaca bukti
RQ1 pada paket ini adalah audit data rilis, bukan pengulangan pengumpulan dan anotasi.
RQ2 membandingkan ViT, IndoBERTweet, dan fusi keduanya. RQ4 membandingkan single-task dengan
dua model multitask terpilih. RQ3 dibahas sebagai referensi paper dan riwayat percobaan,
bukan hasil zero-shot yang berhasil diselesaikan.

Setiap bagian memisahkan **hasil teramati**, **interpretasi sementara**, dan **batas klaim**.
Model dilatih pada fit, checkpoint dipilih dari validation, lalu protokol dikunci sebelum test.
Skor test tidak menjadi alasan menambah seed, mengubah lambda, atau mencari konfigurasi baru.

In [ ]:
display(pd.DataFrame([
 {'Bagian':'RQ1','Cakupan':'Audit data, label, topik, duplikat dan tokenisasi','Status':'Bukti tersimpan'},
 {'Bagian':'RQ2','Cakupan':'Gambar, teks, fusi pada ViT/IndoBERTweet','Status':'6 konfigurasi inti selesai'},
 {'Bagian':'RQ4','Cakupan':'Multitask teks dan fusi','Status':'2 konfigurasi inti selesai'},
 {'Bagian':'RQ3','Cakupan':'Zero-shot/API','Status':'Di luar target aktif, tidak ada hasil lengkap'},
]))
records=[]
for file in inventory.get('staged_files',[]):
    p=path(file['relative_path'])
    assert p.is_file() and p.stat().st_size==file['bytes']
    assert sha256(p)==file['sha256'],file['relative_path']
    records.append({'Berkas':file['relative_path'],'Byte':file['bytes'],'SHA256 cocok':True})
assert records,'Manifest bukti tambahan kosong.'
CHECKS['additional_artifact_hashes']=len(records)
display(pd.DataFrame(records))
print('Seluruh bukti tambahan cocok checksum. Bobot model besar tidak diperlukan untuk notebook ini.')

## Data asal revisi dan lisensi
INDOMEME menggunakan 5.023 meme pada rilis yang dipatok. Materi tambahan penerbit diperiksa silang
dengan rilis Hugging Face. Dataset card menyatakan CC BY 4.0. Atribusi dan sumber dipertahankan.
Label internal `appropriate` menggunakan **1 = inappropriate** dan **0 = appropriate**.
Untuk tugas hate, **1 = hate** dan **0 = not hate**. Nama internal tidak mengubah makna kelas positif.

In [ ]:
meta=read_csv('data/metadata.csv')
assert len(meta)==5023 and meta.image_path.is_unique
assert set(meta.split)=={'train','test'}
assert set(meta.hate_final)=={'hate','not hate'}
assert set(meta.appropriate_final)=={'appropriate','inappropriate'}
assert not ((meta.hate_final=='hate')&(meta.appropriate_final=='appropriate')).any()
assert ((meta.hate_final=='hate').astype(int)==meta.y_hate.astype(int)).all()
assert ((meta.appropriate_final=='inappropriate').astype(int)==meta.y_appropriate.astype(int)).all()
display(pd.DataFrame(facts['sources']))
display(pd.crosstab(meta.split,[meta.hate_final,meta.appropriate_final],margins=True))
CHECKS['metadata_rows']=len(meta)

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,3.7))
for ax,column,title in zip(axes,['hate_final','appropriate_final'],['Label hate','Label kelayakan']):
    counts=pd.crosstab(meta.split,meta[column])
    counts.plot.bar(ax=ax,color=['#264B68','#2B7A78'],rot=0)
    ax.set_title(title);ax.set_xlabel('Split resmi');ax.set_ylabel('Jumlah meme')
    for container in ax.containers:ax.bar_label(container,padding=2,fontsize=8)
fig.tight_layout();plt.show()
print('Macro-F1 memberi bobot setara ke setiap kelas, sedangkan accuracy menimbang seluruh baris.')

### Missing field dan anotasi topik
OCR dan caption berasal dari rilis. Tidak ada regenerasi OCR/caption di eksperimen ini.
Topik kosong menjadi string kosong untuk input model, sedangkan `none/others` dipakai pada tabulasi.
Topik merupakan anotasi tambahan yang belum tentu tersedia ketika sistem menerima meme baru.
Karena itu, hasil dengan topik tidak mewakili pipeline otomatis pengenalan topik.

In [ ]:
missing=[]
for column in ['hate_final','appropriate_final','ocr','caption','topic']:
    values=meta[column].astype(str).str.strip()
    for split in ['train','test']:
        count=int((values[meta.split==split].isin(['','nan','None'])).sum())
        missing.append({'Field':column,'Split':split,'Kosong atau nan':count})
display(pd.DataFrame(missing))
assert not meta.ocr.astype(str).str.strip().isin(['','nan']).any()
assert not meta.caption.astype(str).str.strip().isin(['','nan']).any()
topic_counts=pd.Series(facts['dataset']['topic_counts']).sort_values()
ax=topic_counts.plot.barh(figsize=(8,3.8),color='#264B68')
ax.set_xlabel('Jumlah anotasi, topik dapat multilabel');ax.set_title('Topik rilis seluruh dataset')
plt.tight_layout();plt.show()

### Split dan duplikat
Train resmi 4.018 dibagi menjadi fit 3.214 dan validation 804 dengan fold pertama
StratifiedGroupKFold lima fold, shuffle, seed 42, dan stratifikasi label gabungan.
Tiga pasang gambar identik di train dikelompokkan. Dua kelompok gambar identik tetap melintasi
train/test resmi. Pemisahan ID tidak sama dengan ketiadaan duplikat visual.

In [ ]:
split_sets={}
for identifier in core.id.unique():
    fit=set(read_csv(f'runs/{identifier}/fit_ids.csv').image_path)
    val=set(read_csv(f'runs/{identifier}/validation_ids.csv').image_path)
    pred=read_csv(f'runs/{identifier}/test_predictions.csv')
    test=set(pred.image_path)
    assert len(fit)==3214 and len(val)==804 and len(test)==1005
    assert not fit&val and not fit&test and not val&test
    assert fit|val==set(meta.loc[meta.split=='train','image_path'])
    assert test==set(meta.loc[meta.split=='test','image_path'])
    split_sets[identifier]=(fit,val)
first=next(iter(split_sets.values()))
assert all(value==first for value in split_sets.values())
display(pd.DataFrame([{'Bagian':'Fit','Jumlah':3214},{'Bagian':'Validation','Jumlah':804},{'Bagian':'Test','Jumlah':1005}]))
display(pd.json_normalize(facts['dataset']['duplicate_audit']))
CHECKS['split_ids_eight_runs_checked']=True
print('ID fit/validation identik pada delapan run. Lima kelompok gambar identik meliputi dua kelompok lintas split resmi.')

In [ ]:
phash=read_json('evidence/data_audits/perceptual_audit.json')
phash_pairs=read_csv('evidence/data_audits/perceptual_duplicate_candidates.csv')
assert phash['screening_completed'] and len(phash_pairs)==phash['candidate_pairs']==197
assert phash['cross_split_candidate_pairs']==73
display(pd.DataFrame([phash]))
ax=phash_pairs.groupby('hamming_distance').size().plot.bar(color='#2B7A78',figsize=(7,3.1))
ax.set_xlabel('Jarak Hamming pHash kandidat');ax.set_ylabel('Jumlah pasangan');ax.set_title('Screening kemiripan visual tersimpan')
plt.tight_layout();plt.show()
print('Screening pHash telah dijalankan pada 23 Oktober: 197 pasangan, 73 lintas split, 192 nonexact.')
print('Kandidat pHash adalah sinyal kemiripan. Review manual belum selesai dan split tidak diubah.')

## Preprocessing aktual dan panjang input
Varian bernama Caption+OCR+Topic menggabungkan field tidak kosong dalam urutan **topic, caption, OCR**.
IndoBERTweet memakai separator tokenizer `[SEP]`. Teks dipad/truncate ke 256 token. Pipeline
transformer tidak memakai stemming atau penghapusan stop word. Gambar dibaca sebagai RGB dan
diproses oleh image processor revisi ViT 224 yang dipatok.

In [ ]:
tokens=read_csv('evidence/data_audits/tokenizer_audit_all.csv')
assert len(tokens)==18
display(tokens)
selected_tokens=tokens[(tokens.model=='IndoBERTweet')&(tokens.variant=='Caption+OCR+Topic')]
assert len(selected_tokens)==1
display(selected_tokens)
print('Audit token memakai train resmi; tidak mengunduh atau memanggil tokenizer lagi pada notebook ini.')
fig,axes=plt.subplots(1,2,figsize=(11,3.5))
tokens.pivot(index='variant',columns='model',values='max_tokens').plot.bar(ax=axes[0])
axes[0].axhline(256,color='#BE7E35',linestyle='--',label='Batas256');axes[0].set_ylabel('Maksimum token');axes[0].tick_params(axis='x',rotation=45)
for column in ['ocr','caption']:
    lengths=meta.loc[meta.split=='train',column].astype(str).str.split().str.len()
    axes[1].hist(lengths,bins=30,alpha=.6,label=column)
axes[1].set_xlabel('Panjang kata train, bukan token');axes[1].set_ylabel('Jumlah meme');axes[1].legend()
fig.tight_layout();plt.show()

## Kode implementasi yang digunakan
Bagian berikut menampilkan definisi aktual dari source yang disertakan, tanpa mengimpor atau
memanggil fungsi pelatihan. Dengan demikian, pembaca dapat mengaudit alur pemrosesan dan optimisasi
tanpa menjalankan fine-tuning lagi. Revisi sumber/konfigurasi per run tetap menjadi bukti otoritatif.

In [ ]:
def show_source(relative,names):
    source=path(relative).read_text(encoding='utf-8-sig')
    tree=ast.parse(source)
    selected=[node for node in tree.body if isinstance(node,(ast.FunctionDef,ast.ClassDef)) and node.name in names]
    assert {node.name for node in selected}==set(names)
    display(Code((chr(10)*2).join(ast.get_source_segment(source,node) for node in selected),language='python'))
    print('Sumber:',relative,'SHA256:',sha256(path(relative)))

show_source('src/master_training.py',['text_values','MemeDataset'])

In [ ]:
show_source('src/master_training.py',['MemeClassifier'])
print('Representasi pooler IndoBERTweet dan ViT dikonkatenasi untuk fusi. Projection linear mengecilkan dimensi menjadi setengah, lalu dropout dan head klasifikasi.')
print('Multitask berbagi representasi dan mempunyai dua head. Model ini tidak menambahkan modul cross-attention baru.')
print('Encoder di-fine-tune pada eksperimen yang telah selesai, bukan dibekukan sebagai pengganti metode.')

In [ ]:
show_source('src/master_training.py',['total_loss','evaluate'])
print('Loss multitask = 0,5 cross-entropy hate + 0,5 cross-entropy inappropriate.')
print('Aturan constrained: prediksi inappropriate = maksimum(prediksi raw inappropriate, prediksi hate).')
print('Raw disimpan; evaluasi inappropriate multitask utama memakai constrained.')

In [ ]:
show_source('src/master_training.py',['run_training','lock_test_protocol','evaluate_official_test'])
print('Definisi di atas hanya ditampilkan. Tidak satu pun fungsi training atau inference dipanggil.')

### Protokol dan deviasi
Seed 42, maksimum 5 epoch, AdamW, learning rate 2e-5, weight decay 0,01, dropout 0,3, precision FP32,
warmup 0, dan patience 2. Batch historis 2 × akumulasi 4 serta lab 1 × akumulasi 8 memberi batch efektif
sekitar 8, tetapi tidak menjamin trajektori optimisasi identik.

Checkpoint single-task dipilih dari Macro-F1 validation. Multitask memakai rata-rata hate dan
constrained inappropriate. History dan resume checkpoint disimpan tiap epoch. Prediksi/metrik
validation serta best checkpoint diperbarui saat skor seleksi membaik. Test memakai checkpoint
yang sudah terkunci. Lambda 0,5 belum mereproduksi proses tuning lambda penulis.

In [ ]:
configuration_rows=[]
for identifier in core.id.unique():
    cfg=read_json(f'runs/{identifier}/config.json')
    assert cfg['seed']==42 and cfg['scope']=='full' and cfg['max_train'] is None and cfg['max_validation'] is None
    configuration_rows.append({'ID':identifier,'Kind':cfg['kind'],'Task':cfg['task'],
        'Physical batch':cfg['batch_size'],'Accumulation':cfg['gradient_accumulation'],
        'Precision':cfg['precision'],'LR':cfg['learning_rate'],'Selection':cfg['selection_metric']})
display(pd.DataFrame(configuration_rows))
display(pd.DataFrame(facts['protocol'].items(),columns=['Parameter','Nilai']))
deviations=[]
for run in facts['runs']:
    for note in run['config']['deviations']:
        if note not in deviations:deviations.append(note)
display(pd.DataFrame({'Deviasi yang tercatat':deviations}))

## Register status seluruh pekerjaan
Jumlah konfigurasi inti, keluaran tugas, baseline validation, smoke test, dan percobaan gagal
dilaporkan terpisah. Satu model multitask menghasilkan dua hasil tugas. Kalibrasi satu batch
tidak dihitung sebagai training penuh. Riwayat gagal tidak menghapus keberhasilan run berikutnya.

In [ ]:
register=[]
for run in facts['runs']:
    register.append({'ID':run['id'],'Kelompok':'Paket inti','Status':'Completed test','Epoch':run['epochs_trained'],
                     'Hasil tugas':2 if run['config']['task']=='both' else 1})
for run in inventory['supplementary_supervised']:
    register.append({'ID':run['id'],'Kelompok':'Supervised historis di luar paket inti','Status':run['status'],
                     'Epoch':len(run.get('history',[])),'Hasil tugas':1 if run.get('test_metrics') else 0})
display(pd.DataFrame(register))
display(pd.DataFrame([{'Jenis':'Konfigurasi inti selesai','Jumlah':8},{'Jenis':'Hasil tugas inti','Jumlah':10},
 {'Jenis':'Baseline validation tambahan','Jumlah':len(inventory['baseline_trials'])},
 {'Jenis':'Kalibrasi atau smoke record','Jumlah':len(inventory.get('calibrations',[]))}]))

## Baseline konvensional validation
Empat belas baseline terdiri dari Majority serta enam varian TF-IDF dan logistic regression
untuk dua tugas. Pengujian menggunakan validation 804 dari train resmi. Hasil ini adalah sanity
baseline tambahan, bukan reproduksi model transformer paper dan bukan skor test 1.005.
Vectorizer dipasang hanya pada fit. Tidak ada training ulang baseline di notebook ini.

In [ ]:
baseline_config=read_json(inventory['baseline_trials'][0]['config_path'])
display(pd.DataFrame([{'Parameter':key,'Nilai':baseline_config[key]} for key in
 ['evaluation','seed','split','fit_rows','validation_rows','separator','tfidf','logreg','selection','topic_note']]))
show_source('src/baseline_text.py',['partition','text_input','run_baselines'])
print('Definisi baseline hanya diperlihatkan. Fit logistic regression tidak dijalankan ulang.')

In [ ]:
def manual_metrics(truth,prediction):
    truth=np.asarray(truth,dtype=int);prediction=np.asarray(prediction,dtype=int)
    assert len(truth)==len(prediction)>0 and set(truth)<= {0,1} and set(prediction)<= {0,1}
    cm=np.zeros((2,2),dtype=int)
    for y,pred in zip(truth,prediction):cm[y,pred]+=1
    per=[]
    for label in [0,1]:
        tp=int(cm[label,label]);support=int(cm[label].sum());predicted=int(cm[:,label].sum())
        precision=tp/predicted if predicted else 0.;recall=tp/support if support else 0.
        f1=2*precision*recall/(precision+recall) if precision+recall else 0.
        per.append({'label':label,'precision':precision,'recall':recall,'f1':f1,'support':support})
    result={'macro_precision':np.mean([r['precision'] for r in per]),'macro_recall':np.mean([r['recall'] for r in per]),
      'macro_f1':np.mean([r['f1'] for r in per]),'accuracy':float(np.trace(cm)/cm.sum()),
      'confusion_matrix_labels_0_1':cm.tolist(),'per_class':per,'rows':len(truth)}
    sk=precision_recall_fscore_support(truth,prediction,labels=[0,1],average='macro',zero_division=0)
    assert np.allclose([result['macro_precision'],result['macro_recall'],result['macro_f1']],sk[:3],atol=1e-12,rtol=0)
    assert abs(result['accuracy']-accuracy_score(truth,prediction))<1e-12
    return result

assert manual_metrics([0,0,1,1],[0,0,0,0])['macro_f1']==1/3
assert manual_metrics([0,1],[0,1])['macro_f1']==1
baseline_rows=[]
for trial in inventory['baseline_trials']:
    frame=read_csv(trial['predictions_path'])
    for key,value in trial['prediction_filter'].items():frame=frame[frame[key]==value]
    assert set(frame.image_path)==first[1] and frame.image_path.is_unique
    truth_column='y_hate' if trial['task']=='hate' else 'y_appropriate'
    assert np.array_equal(frame.truth.astype(int),meta.set_index('image_path').loc[frame.image_path,truth_column].astype(int).to_numpy())
    result=manual_metrics(frame.truth,frame.prediction)
    assert result['rows']==trial['rows']==804
    for key in ['macro_precision','macro_recall','macro_f1','accuracy']:
        assert abs(result[key]-trial['metrics'][key])<1e-12
    baseline_rows.append({'ID':trial['trial_id'],'Task':trial['task'],'Variant':trial['variant'],
      **{key:result[key] for key in ['macro_precision','macro_recall','macro_f1','accuracy']}})
    CHECKS['baseline_validation_trials']+=1
baseline=pd.DataFrame(baseline_rows)
assert len(baseline)==14
display(baseline)

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4),sharey=True)
for ax,task in zip(axes,['hate','appropriate']):
    subset=baseline[baseline.Task==task]
    bars=ax.barh(subset.Variant,subset.macro_f1,color='#264B68')
    ax.bar_label(bars,fmt='%.4f',padding=3);ax.set_xlim(0,1)
    ax.set_title(task+' validation804');ax.set_xlabel('Macro-F1 validation')
fig.tight_layout();plt.show()
for trial in inventory['baseline_trials']:
    display(Markdown(f"### {trial['trial_id']}  {trial['task']}  {trial['variant']}"))
    m=trial['metrics']
    display(pd.DataFrame([{'ID':trial['trial_id'],'Tugas':trial['task'],'Varian':trial['variant'],
       'Evaluasi':'validation','Jumlah':trial['rows'],
       **{key:m[key] for key in ['macro_precision','macro_recall','macro_f1','accuracy']}}]))
    display(pd.DataFrame(m['per_class']))
    cm=np.array(m['confusion_matrix_labels_0_1'])
    fig,ax=plt.subplots(figsize=(3.3,2.8));ax.imshow(cm,cmap='Blues')
    for i in range(2):
        for j in range(2):ax.text(j,i,str(cm[i,j]),ha='center',va='center',color='white' if cm[i,j]>cm.max()/2 else 'black')
    ax.set(xticks=[0,1],yticks=[0,1],xlabel='Prediksi',ylabel='Truth',title=trial['trial_id']+' validation')
    fig.tight_layout();plt.show()
    error_rows=[]
    for kind,label in [('false_positive','False positive'),('false_negative','False negative')]:
        examples=trial.get('error_examples',{}).get(kind,[])
        if not examples:print('Tidak ada contoh '+label.lower()+' pada artefak trial ini.')
        for example in examples:
            error_rows.append({'Jenis':label,'ID gambar':example['image_path'],
                               'Truth':example['truth'],'Prediksi':example['prediction']})
    if error_rows:display(pd.DataFrame(error_rows))
    print(f"Insight {trial['trial_id']}: Macro-F1 validation {m['macro_f1']:.4f}, accuracy {m['accuracy']:.4f}, FP {cm[0,1]}, dan FN {cm[1,0]}.")
    if trial['variant']=='Majority':
        print('Model mayoritas memprediksi kelas 0. Recall kelas positif bernilai nol, sehingga accuracy tidak cukup untuk menilai deteksi kelas positif.')
    else:
        reference_variant={'OCR':'Majority','Caption':'OCR','Caption+OCR':'Caption',
            'Caption+Topic':'Caption','OCR+Topic':'OCR','Caption+OCR+Topic':'Caption+OCR'}[trial['variant']]
        reference=next(item for item in inventory['baseline_trials']
                       if item['task']==trial['task'] and item['variant']==reference_variant)
        difference=100*(m['macro_f1']-reference['metrics']['macro_f1'])
        print(f"Pada validation yang sama, selisih Macro-F1 terhadap {reference_variant} adalah {difference:+.4f} poin persentase.")
        if 'Topic' in trial['variant']:
            print('Varian ini memakai topik anotasi rilis. Hasil tidak mewakili sistem yang harus memprediksi topik meme baru.')
        else:print('Varian ini tidak memakai anotasi topik.')
    print('Hasil hanya berlaku pada validation 804 dan satu seed. Contoh error bersifat deterministik, tanpa anotasi penyebab semantik atau klaim signifikansi.')

## Analisis per eksperimen inti
Bagian berikut membahas setiap ID secara utuh: konfigurasi dan riwayat, evaluasi validation/test,
confusion matrix, kelas positif, contoh kesalahan, perbandingan paper, serta insight teramati.
Angka dihitung ulang dari CSV prediksi yang dibekukan. Perbedaan validation/test tidak dengan
sendirinya membuktikan penyebab overfitting atau kualitas anotasi.

In [ ]:
RUN_RESULTS={}

def prediction_result(identifier,split,task,policy):
    frame=read_csv(f'runs/{identifier}/{split}_predictions.csv')
    frame=frame[frame.task==task].copy()
    assert len(frame)==(1005 if split=='test' else 804) and frame.image_path.is_unique
    labels=meta.set_index('image_path')['y_hate' if task=='hate' else 'y_appropriate'].astype(int)
    assert np.array_equal(frame.truth.astype(int),labels.loc[frame.image_path].to_numpy())
    column='prediction_constrained' if policy=='constrained' else 'prediction'
    assert column in frame
    frame['evaluated_prediction']=frame[column].astype(int)
    return manual_metrics(frame.truth,frame.evaluated_prediction),frame

def plot_cm_panels(records):
    rows=(len(records)+1)//2
    fig,axes=plt.subplots(rows,2,figsize=(8,3.3*rows),squeeze=False)
    for ax,(title,result) in zip(axes.flat,records):
        cm=np.array(result['confusion_matrix_labels_0_1']);ax.imshow(cm,cmap='Blues',vmin=0,vmax=650)
        for i in range(2):
            for j in range(2):ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=13,color='white' if cm[i,j]>350 else '#172733')
        ax.set(xticks=[0,1],yticks=[0,1],xlabel='Prediksi',ylabel='Truth',title=title)
    for ax in list(axes.flat)[len(records):]:ax.axis('off')
    fig.tight_layout();plt.show()

def describe_run(identifier):
    config=read_json(f'runs/{identifier}/config.json')
    result=read_json(f'runs/{identifier}/result.json')
    history=read_csv(f'runs/{identifier}/history.csv')
    reference=next(r for r in facts['runs'] if r['id']==identifier)
    best=history.loc[history.selection_macro_f1.astype(float).idxmax()]
    assert int(best.epoch)==reference['selected_epoch']
    display(pd.DataFrame([{'ID':identifier,'Run ID':result.get('run_id',reference['config']['experiment_id']),
       'Kind':config['kind'],'Task':config['task'],'Batch':config['batch_size'],
       'Akumulasi':config['gradient_accumulation'],'Epoch jalan':len(history),'Epoch dipilih':int(best.epoch),
       'Config SHA256':sha256(path(f'runs/{identifier}/config.json'))}]))
    display(pd.json_normalize(config['models']))
    display(history)
    fig,axes=plt.subplots(1,2,figsize=(10,3.1))
    axes[0].plot(history.epoch,history.train_loss,marker='o',color='#2B7A78');axes[0].set(xlabel='Epoch',ylabel='Train loss',title=identifier+' loss')
    axes[1].plot(history.epoch,history.selection_macro_f1,marker='o',label='Skor seleksi validation')
    axes[1].axvline(int(best.epoch),ls='--',color='#BE7E35',label='Checkpoint pilihan')
    axes[1].set(xlabel='Epoch',ylabel='Macro-F1 seleksi',title=identifier+' validation');axes[1].legend(fontsize=8)
    for ax in axes:ax.set_xticks(history.epoch);ax.grid(alpha=.15)
    fig.tight_layout();plt.show()
    print('Status: completed test. Epoch yang dijalankan dan epoch checkpoint terpilih dibedakan.')

def evaluate_run(identifier):
    relevant=[o for o in facts['outcomes'] if o['id']==identifier]
    metrics_rows=[];classes=[];panels=[]
    for outcome in relevant:
        task=outcome['task'];policy=outcome['policy']
        for split in ['validation','test']:
            computed,frame=prediction_result(identifier,split,task,policy)
            expected=outcome['validation'] if split=='validation' else outcome['test']
            for key in ['macro_precision','macro_recall','macro_f1','accuracy']:
                assert abs(computed[key]-expected[key])<1e-12,(identifier,task,split,key)
            RUN_RESULTS[(identifier,task,split)]=(computed,frame)
            metrics_rows.append({'Split':split,'Tugas':task,'Policy':policy,**{k:computed[k] for k in ['macro_precision','macro_recall','macro_f1','accuracy']}})
            classes.extend({'Split':split,'Task':task,**r} for r in computed['per_class'])
            panels.append((identifier+' '+task+' '+split,computed))
            if split=='test':CHECKS['core_test_outcomes']+=1
        if policy=='constrained':
            raw,_=prediction_result(identifier,'test',task,'raw')
            metrics_rows.append({'Split':'test tambahan','Tugas':task,'Policy':'raw',**{k:raw[k] for k in ['macro_precision','macro_recall','macro_f1','accuracy']}})
    display(pd.DataFrame(metrics_rows));display(pd.DataFrame(classes))
    plot_cm_panels(panels)
    print('Baris=truth, kolom=prediksi. Kelas1=hate pada hate dan inappropriate pada appropriate.')

def inspect_errors(identifier):
    for outcome in [o for o in facts['outcomes'] if o['id']==identifier]:
        task=outcome['task'];m,frame=RUN_RESULTS[(identifier,task,'test')]
        errors=frame[frame.truth.astype(int)!=frame.evaluated_prediction].copy()
        errors['Jenis']=np.where(errors.truth.astype(int)==0,'false positive','false negative')
        examples=pd.concat([group.sort_values('image_path').head(3) for _,group in errors.groupby('Jenis')])
        display(Markdown('#### '+identifier+' '+task+' contoh kesalahan'))
        display(examples[['image_path','truth','evaluated_prediction','Jenis']])
        joined=frame.merge(meta[['image_path','joint_label']],on='image_path',validate='one_to_one')
        joined['correct']=(joined.truth.astype(int)==joined.evaluated_prediction)
        display(joined.groupby('joint_label').agg(Jumlah=('correct','size'),Benar=('correct','sum'),Proporsi_benar=('correct','mean')))
        cm=m['confusion_matrix_labels_0_1'];positive=m['per_class'][1]
        print(f"{identifier} {task}: FP={cm[0][1]}, FN={cm[1][0]}, recall kelas1={positive['recall']:.4f}.")
        print(f"Macro-F1 validation={outcome['validation_macro_f1']:.4f}, test={m['macro_f1']:.4f}, paper={outcome['paper_macro_f1']:.3f}, delta={outcome['delta_percentage_points']:+.2f} pp.")
        print('Contoh dipilih deterministik untuk pelacakan, bukan sampel prevalensi penyebab kesalahan.')

### E005 Gambar ViT untuk hate

Gambar saja memberi baseline modalitas. Recall hate yang teramati dibandingkan dengan model teks, tanpa menyimpulkan bahwa penyebabnya selalu teks di dalam meme.

In [ ]:
describe_run('E005')

In [ ]:
evaluate_run('E005')

In [ ]:
inspect_errors('E005')
print('Gambar saja memberi baseline modalitas. Recall hate yang teramati dibandingkan dengan model teks, tanpa menyimpulkan bahwa penyebabnya selalu teks di dalam meme.')

### E006 Gambar ViT untuk inappropriate

Run lab berhenti normal pada epoch4 setelah checkpoint terbaik epoch2. Ini early stopping, bukan training yang gagal.

In [ ]:
describe_run('E006')

In [ ]:
evaluate_run('E006')

In [ ]:
inspect_errors('E006')
print('Run lab berhenti normal pada epoch4 setelah checkpoint terbaik epoch2. Ini early stopping, bukan training yang gagal.')

### E041 IndoBERTweet untuk hate

Input memakai topik, caption, dan OCR. Model teks ini menjadi pembanding fusi hate, dengan konteks topik anotasi yang sama.

In [ ]:
describe_run('E041')

In [ ]:
evaluate_run('E041')

In [ ]:
inspect_errors('E041')
print('Input memakai topik, caption, dan OCR. Model teks ini menjadi pembanding fusi hate, dengan konteks topik anotasi yang sama.')

### E042 IndoBERTweet untuk inappropriate

Kandidat berdasarkan validation per tugas. Kandidat ini tidak dipilih karena memiliki test tertinggi. Margin validation terhadap E110 sangat kecil.

In [ ]:
describe_run('E042')

In [ ]:
evaluate_run('E042')

In [ ]:
inspect_errors('E042')
print('Kandidat berdasarkan validation per tugas. Kandidat ini tidak dipilih karena memiliki test tertinggi. Margin validation terhadap E110 sangat kecil.')

### E059 Fusi ViT dan IndoBERTweet untuk hate

Run yang berhasil merupakan kelanjutan setelah kegagalan OOM terdahulu dengan konfigurasi eksekusi baru. Fusi tidak memperbaiki Macro-F1 test hate dibanding E041 pada seed ini.

In [ ]:
describe_run('E059')

In [ ]:
evaluate_run('E059')

In [ ]:
inspect_errors('E059')
print('Run yang berhasil merupakan kelanjutan setelah kegagalan OOM terdahulu dengan konfigurasi eksekusi baru. Fusi tidak memperbaiki Macro-F1 test hate dibanding E041 pada seed ini.')

### E060 Fusi ViT dan IndoBERTweet untuk inappropriate

Macro-F1 test inappropriate tertinggi pada paket inti teramati pada E060. Skor validation-nya lebih rendah daripada E042, sehingga keduanya tidak boleh dipertukarkan sebagai pemenang test dan kandidat validation.

In [ ]:
describe_run('E060')

In [ ]:
evaluate_run('E060')

In [ ]:
inspect_errors('E060')
print('Macro-F1 test inappropriate tertinggi pada paket inti teramati pada E060. Skor validation-nya lebih rendah daripada E042, sehingga keduanya tidak boleh dipertukarkan sebagai pemenang test dan kandidat validation.')

### E101 Multitask teks

Satu training menghasilkan dua tugas. Checkpoint dipilih dari rata-rata metrik validation kedua tugas. Raw dan constrained inappropriate disimpan terpisah.

In [ ]:
describe_run('E101')

In [ ]:
evaluate_run('E101')

In [ ]:
inspect_errors('E101')
print('Satu training menghasilkan dua tugas. Checkpoint dipilih dari rata-rata metrik validation kedua tugas. Raw dan constrained inappropriate disimpan terpisah.')

### E110 Multitask multimodal

Satu training menghasilkan dua tugas. Epoch5 menyelesaikan training, sedangkan epoch 3 menjadi checkpoint terpilih. Multitask tidak otomatis memperbaiki kedua tugas.

In [ ]:
describe_run('E110')

In [ ]:
evaluate_run('E110')

In [ ]:
inspect_errors('E110')
print('Satu training menghasilkan dua tugas. Epoch5 menyelesaikan training, sedangkan epoch3 menjadi checkpoint terpilih. Multitask tidak otomatis memperbaiki kedua tugas.')

## Supervised historis di luar paket inti
SwinV2 E001 dan E002 berasal dari antrean historis yang lebih luas. E001 mempunyai hasil test;
E002 menyimpan proses parsial. Hasil tambahan dipertahankan agar seluruh progres dapat dilacak,
tanpa mengubah definisi paket inti 8 konfigurasi / 10 hasil dan tanpa menjadikan proses parsial sebagai
hasil test. Penyebutan epoch parsial tidak menjamin run tersebut layak dilanjutkan tanpa preflight baru.

In [ ]:
for extra in inventory['supplementary_supervised']:
    display(Markdown('### '+extra['id']+' '+extra['status']))
    cfg=extra['config']
    display(pd.DataFrame([{'Run ID':extra.get('run_id',''),'Task':cfg['task'],'Encoder gambar':cfg['image'],
        'Batch':cfg['batch_size'],'Akumulasi':cfg['gradient_accumulation'],'Status':extra['status']}]))
    display(pd.DataFrame(extra.get('history',[])))
    print('Status test:', 'ada' if extra.get('test_metrics') else 'belum dievaluasi')
    print('Catatan:',extra.get('insight',extra.get('note','Di luar paket inti.')))
    panels=[];extra_metrics=[]
    for split in ['validation','test']:
        pred_path=extra.get('files',{}).get(split+'_predictions')
        if not pred_path:continue
        frame=read_csv(pred_path)
        for task,group in frame.groupby('task'):
            result=manual_metrics(group.truth,group.prediction)
            expected=extra[split+'_metrics'][task]
            for key in ['macro_precision','macro_recall','macro_f1','accuracy']:
                assert abs(result[key]-expected[key])<1e-12
            extra_metrics.append({'Split':split,'Task':task,**{key:result[key] for key in ['macro_precision','macro_recall','macro_f1','accuracy']}})
            display(pd.DataFrame(result['per_class']).assign(Split=split,Tugas=task))
            panels.append((extra['id']+' '+task+' '+split,result))
            errors=group[group.truth.astype(int)!=group.prediction.astype(int)].sort_values('image_path').head(6)
            display(errors[['image_path','truth','prediction']])
    display(pd.DataFrame(extra_metrics))
    if panels:plot_cm_panels(panels)
    history=pd.DataFrame(extra.get('history',[]))
    if not history.empty and 'epoch' in history:
        fig,ax=plt.subplots(figsize=(6,2.7))
        ax.plot(history.epoch.astype(int),history.selection_macro_f1.astype(float),marker='o')
        ax.set(xlabel='Epoch tersimpan',ylabel='Skor validation',title=extra['id']+' riwayat historis')
        fig.tight_layout();plt.show()

## Dampak aturan hierarki multitask
Aturan hate mengimplikasikan inappropriate meningkatkan konsistensi label, tetapi belum tentu
meningkatkan Macro-F1. Di bawah ini setiap perubahan prediksi ditelusuri langsung dari kolom raw
dan constrained pada validation serta test. Probabilitas head bukan probabilitas yang telah
dikalibrasi dan tidak digunakan untuk mengganti threshold.

In [ ]:
hierarchy=[]
for identifier in ['E101','E110']:
    for split in ['validation','test']:
        frame=read_csv(f'runs/{identifier}/{split}_predictions.csv')
        a=frame[frame.task=='appropriate'].set_index('image_path')
        h=frame[frame.task=='hate'].set_index('image_path').loc[a.index]
        expected=np.maximum(a.prediction.astype(int),h.prediction.astype(int))
        assert np.array_equal(expected,a.prediction_constrained.astype(int))
        raw=manual_metrics(a.truth,a.prediction);constrained=manual_metrics(a.truth,a.prediction_constrained)
        changed=a.prediction.astype(int)!=a.prediction_constrained.astype(int)
        improved=changed&(a.truth.astype(int)==a.prediction_constrained.astype(int))
        worsened=changed&(a.truth.astype(int)==a.prediction.astype(int))
        hierarchy.append({'ID':identifier,'Split':split,'Raw F1':raw['macro_f1'],'Constrained F1':constrained['macro_f1'],
          'Berubah':int(changed.sum()),'Salah menjadi benar':int(improved.sum()),'Benar menjadi salah':int(worsened.sum())})
        if changed.any():display(a.loc[changed,['truth','prediction','prediction_constrained']])
display(pd.DataFrame(hierarchy))
print('Interpretasi: relasi hierarki dan skor per kelas mengukur hal berbeda. E101 test mengubah satu prediksi benar menjadi salah.')

## Perbandingan berpasangan modalitas dan multitask
Perbandingan memakai ID test yang sama. A benar saja dan B benar saja menunjukkan pertukaran
kesalahan, sedangkan keduanya salah menunjukkan sasaran audit manual bersama. Selisih Macro-F1
bersifat deskriptif pada seed 42. Notebook tidak mengklaim signifikansi atau penyebab semantik.

In [ ]:
pairs=[]
for reference in facts['comparisons']:
    task=reference['task'];a=reference['a'];b=reference['b']
    ma,fa=RUN_RESULTS[(a,task,'test')];mb,fb=RUN_RESULTS[(b,task,'test')]
    joined=fa[['image_path','truth','evaluated_prediction']].merge(fb[['image_path','truth','evaluated_prediction']],on='image_path',suffixes=('_a','_b'),validate='one_to_one')
    assert (joined.truth_a==joined.truth_b).all() and len(joined)==1005
    ca=joined.truth_a.astype(int)==joined.evaluated_prediction_a;cb=joined.truth_b.astype(int)==joined.evaluated_prediction_b
    row={'A':a,'B':b,'Task':task,'Keduanya benar':int((ca&cb).sum()),'A benar saja':int((ca&~cb).sum()),
      'B benar saja':int((~ca&cb).sum()),'Keduanya salah':int((~ca&~cb).sum()),'Delta B pp':100*(mb['macro_f1']-ma['macro_f1'])}
    assert row['A benar saja']==reference['a_only_correct'] and row['B benar saja']==reference['b_only_correct']
    pairs.append(row)
display(pd.DataFrame(pairs))
ax=pd.DataFrame(pairs).assign(Pasangan=lambda d:d.A+'/'+d.B+' '+d.Task).plot.barh(x='Pasangan',y='Delta B pp',color='#2B7A78',legend=False,figsize=(9,3.8))
ax.axvline(0,color='#666',lw=.8);ax.set_xlabel('Selisih Macro-F1 B terhadap A, poin persentase')
plt.tight_layout();plt.show()
print('Fusi E059 vs E041 hate:',pairs[0]['Delta B pp'],'pp. Fusi E060 vs E042 inappropriate:',pairs[1]['Delta B pp'],'pp.')

## Kandidat berdasarkan validation
Pemilihan checkpoint di dalam setiap run tetap mengikuti kriteria yang sudah ditetapkan.
Sebagai interpretasi pengumpulan, kandidat per tugas dibandingkan dari validation pada checkpoint
yang sudah terpilih. Test dilaporkan sesudahnya. Ini bukan klaim kandidat yang stabil lintas seed,
dan bukan kebijakan global yang diklaim telah dikunci sebelum semua run historis.

In [ ]:
rankings=[]
for outcome in facts['outcomes']:
    validation,_=RUN_RESULTS[(outcome['id'],outcome['task'],'validation')]
    test,_=RUN_RESULTS[(outcome['id'],outcome['task'],'test')]
    rankings.append({'ID':outcome['id'],'Task':outcome['task'],'Validation F1':validation['macro_f1'],'Test F1':test['macro_f1']})
rankings=pd.DataFrame(rankings)
for task,group in rankings.groupby('Task'):
    ranked=group.sort_values('Validation F1',ascending=False).reset_index(drop=True)
    display(ranked)
    candidate=ranked.iloc[0];margin=100*(ranked.iloc[0]['Validation F1']-ranked.iloc[1]['Validation F1'])
    assert candidate.ID==facts['validation_candidates'][task]['id']
    print(f"Kandidat {task}: {candidate.ID}, margin validation {margin:.5f} pp.")
print('E059 menjadi kandidat hate. E042 menjadi kandidat inappropriate. E060 test tertinggi tidak menggantikan keputusan berbasis validation.')

## Perbandingan dengan paper
Macro-F1 paper merupakan acuan pada kondisi terpilih. Selisih poin persentase adalah 100 kali
skor reproduksi dikurangi skor paper. Perbedaan batch, validation, checkpoint ViT, pooling,
dan tuning lambda membatasi klaim reproduksi numerik identik. Angka lebih tinggi pada satu run
tidak membuktikan keunggulan yang konsisten.

In [ ]:
comparison=core[['id','task','policy','macro_f1','paper_macro_f1','delta_percentage_points']].copy()
assert np.allclose(100*(comparison.macro_f1-comparison.paper_macro_f1),comparison.delta_percentage_points,atol=1e-12,rtol=0)
display(comparison)
fig,axes=plt.subplots(1,2,figsize=(11,3.8),sharey=True)
for ax,task in zip(axes,['hate','appropriate']):
    subset=comparison[comparison.task==task];x=np.arange(len(subset));w=.36
    ax.bar(x-w/2,subset.macro_f1,w,label='Reproduksi',color='#264B68')
    ax.bar(x+w/2,subset.paper_macro_f1,w,label='Paper',color='#BE7E35')
    ax.set(xticks=x,xticklabels=subset.id,ylim=(0,1),title=task,ylabel='Macro-F1 test')
    ax.legend();ax.grid(axis='y',alpha=.15);ax.set_axisbelow(True)
fig.tight_layout();plt.show()

## Kegagalan percobaan dan keputusan penanganan
Riwayat CUDA out of memory, percobaan inferensi RQ3 historis, dan run parsial adalah bagian progres yang relevan.
Statusnya dipisahkan dari hasil final. Isi error yang ditampilkan berasal dari bukti yang sudah
disaring dalam inventory, tanpa kredensial. Tidak ada kegagalan yang diperbaiki dengan mengubah
test atau mengganti metode secara diam-diam.

In [ ]:
display(pd.DataFrame(inventory.get('failed_attempts',[])))
display(pd.DataFrame(inventory.get('rq3_attempts',[])))
print('OOM historis ditangani melalui penyesuaian microbatch dan eksekusi pada lab; konfigurasi run baru dicatat terpisah.')
print('Percobaan RQ3 yang gagal/tidak lengkap tidak menghasilkan skor test yang dapat dipakai untuk klaim supervised versus MLLM.')
print('E002 parsial berada di luar paket aktif; tidak dipromosikan menjadi eksperimen selesai.')

## Kalibrasi dan batas sumber daya
Smoke test satu batch serta kalibrasi VRAM/checkpoint menunjukkan kesiapan tahap tertentu,
bukan hasil training penuh. Penggunaan folder kelompok, tmux, batas ruang, dan antrean berurutan
mendukung keberlangsungan proses pada akun lab bersama. Server tidak dipasangi agentic tools.

In [ ]:
display(pd.DataFrame(inventory.get('calibrations',[])))
display(pd.DataFrame(inventory.get('runtime_checks',[])))
lab_calibration_rows=[]
for identifier in ['E059','E101','E006','E110']:
    calibration=read_json(f'evidence/lab_calibrations/{identifier}.json')
    assert calibration['passed'] is True and calibration['id']==identifier
    assert calibration['optimizer_steps']==1 and calibration['accumulation_steps']==8
    assert calibration['checkpoint_atomic_save_and_load'] is True
    assert calibration['test_evaluated'] is False
    assert calibration['input_length']==(None if identifier=='E006' else 256)
    assert calibration['seconds']>0 and calibration['peak_allocated_mib']>0
    lab_calibration_rows.append({'ID':identifier,'Lulus':True,'GPU':calibration['gpu_name'],
      'Panjang teks':calibration['input_length'],'Langkah optimizer':calibration['optimizer_steps'],
      'Akumulasi':calibration['accumulation_steps'],'VRAM allocated MiB':calibration['peak_allocated_mib'],
      'VRAM reserved MiB':calibration['peak_reserved_mib'],'Durasi detik':calibration['seconds'],
      'Simpan dan muat checkpoint':calibration['checkpoint_atomic_save_and_load'],'Test dievaluasi':False})
display(pd.DataFrame(lab_calibration_rows))
CHECKS['lab_calibrations_checked']=len(lab_calibration_rows)
print('Empat catatan kalibrasi lab ini berasal dari artefak yang diambil melalui SSH. Notebook hanya membaca dan memeriksanya, tanpa menjalankan ulang optimizer.')
print('E006 menerima gambar saja, sehingga panjang teks tidak berlaku. Tiga kondisi teks/fusi memakai panjang input 256 saat kalibrasi.')
print('Training inti berjalan berurutan. Microbatch1/akumulasi8 pada lab menggunakan FP32.')
print('Cadangan operasional laptop: C minimal10GiB dan D minimal12GiB, di luar kebutuhan puncak checkpoint.')
print('Waktu atau VRAM smoke test tidak diekstrapolasi sebagai durasi atau biaya full training.')

## Sintesis hasil dan batas interpretasi
Pada seed 42, fusi tidak memberi perbaikan yang seragam: hate sedikit turun terhadap teks,
sedangkan inappropriate meningkat. Multitask juga belum memperbaiki kedua tugas secara konsisten.
Kelompok not hate tetapi inappropriate berisi 86 meme test, sehingga relasi hierarki hate tidak
mencakup seluruh tantangan kelayakan.

Hasil kuantitatif belum membuktikan penyebab seperti sarkasme, kualitas OCR, konteks budaya, atau
ambiguitas anotasi. Contoh error dan kandidat pHash memerlukan review manual dengan protokol
anotasi. Variasi lintas seed, evaluasi tanpa topik anotasi, dan penanganan duplikat memerlukan
rancangan lanjutan. RQ3/API, grid penuh, dan novelty tidak dinyatakan selesai.

In [ ]:
subgroups=[]
for identifier in ['E042','E060','E101','E110']:
    _,frame=RUN_RESULTS[(identifier,'appropriate','test')]
    joined=frame.merge(meta[['image_path','joint_label']],on='image_path',validate='one_to_one')
    group=joined[joined.joint_label=='not hate | inappropriate']
    assert len(group)==86
    correct=int((group.truth.astype(int)==group.evaluated_prediction).sum())
    subgroups.append({'ID':identifier,'Kelompok':'not hate dan inappropriate','Jumlah':86,'Benar':correct,'Proporsi benar':correct/86})
display(pd.DataFrame(subgroups))
current_limitations=[
 'Paket inti delapan konfigurasi dari 90 supervised. RQ3/API dan pengembangan metode baru berada di luar target aktif.',
 'Satu seed dan satu pembagian validation. Variasi lintas seed, interval kepercayaan, dan klaim signifikansi belum tersedia.',
 'Batch, validation, early stopping, checkpoint ViT dan pooling berbeda atau perlu klarifikasi terhadap implementasi penulis.',
 'Topik anotasi rilis adalah privileged input. Sistem yang harus memprediksi topik meme baru belum dievaluasi.',
 'Dua kelompok gambar identik melintasi train/test resmi. Screening pHash 197 kandidat telah dilakukan, tetapi review manual belum selesai.',
 'Error kuantitatif belum menetapkan penyebab sarkasme, OCR atau konteks budaya tanpa anotasi manual.',
 'Pemilihan kandidat lintas konfigurasi adalah interpretasi validation saat pengumpulan, bukan pencarian model berdasarkan test.'
]
display(pd.DataFrame({'Keterbatasan':current_limitations}))
print('Koreksi cakupan audit: pHash screening sudah dilakukan; yang belum selesai adalah review manual kandidat.')

## Backup reproduksibilitas dan rujukan
Kode, config, history, prediksi, metrik, revisi sumber, dan receipt dipertahankan terpisah dari
bobot besar. Backup utama memuat 8 best checkpoint dan 4 resume checkpoint lab yang dipecah menjadi 73 bagian.
Seluruh bagian serta berkas pendukung telah cocok checksum unduhan pada snapshot 29 Oktober.
Pembukaan CPU dan penggabungan bagian lokal telah lulus. Checksum unduh ulang 8 best checkpoint utuh
di Drive belum diperiksa pada snapshot tersebut. Resume checkpoint historis tidak dipertahankan.

Untuk mengulang analisis notebook, gunakan paket bukti lengkap dan jalankan semua sel pada Python 3.11+
dengan numpy, pandas, matplotlib, scikit-learn, dan kernel IPython. Tidak dibutuhkan GPU, bobot,
PyTorch, koneksi internet, token API, atau password server di notebook ini.

In [ ]:
display(pd.DataFrame(facts['sources']))
print('DOI paper: 10.7717/peerj-cs.3736')
print('Dataset card: https://huggingface.co/datasets/aiatums/INDOMEME')
print('Publisher: https://peerj.com/articles/cs-3736/')
print('Sumber implementasi tampak pada sel model dan training, serta folder src/.')
print('Tidak ada langkah download, training, inferensi baru atau pemanggilan layanan API pada eksekusi notebook ini.')

## Bukti eksekusi notebook
Sel ini hanya dapat menghasilkan keluaran setelah seluruh analisis di atas benar-benar dijalankan.
Host, waktu, jumlah hasil yang diperiksa, serta versi runtime dicatat untuk membedakan notebook
yang sekadar ditulis dari notebook yang dieksekusi. Receipt tidak memuat password atau token.

In [ ]:
assert CHECKS['core_test_outcomes']==10
assert CHECKS['baseline_validation_trials']==14
assert CHECKS['lab_calibrations_checked']==4
assert CHECKS['split_ids_eight_runs_checked'] and CHECKS['metadata_rows']==5023
assert CHECKS['training_or_inference_started'] is False
finished=datetime.now(timezone.utc)
execution_receipt={'status':'all_analysis_cells_completed','executed_utc':finished.isoformat(),
 'executed_wib':finished.astimezone(timezone(timedelta(hours=7))).isoformat(),
 'hostname':socket.gethostname(),'platform':platform.platform(),'python':platform.python_version(),
 'numpy':np.__version__,'pandas':pd.__version__,'checks':CHECKS,
 'mode':'recompute frozen prediction artefacts, no retraining/inference/API/download'}
path('execution_receipt.json').write_text(json.dumps(execution_receipt,indent=2),encoding='utf-8')
display(pd.DataFrame(execution_receipt.items(),columns=['Bukti eksekusi','Nilai']))
print('SELESAI: seluruh sel analisis berjalan dan pemeriksaan wajib lulus.')